In [ ]:
import pandas as pd
import requests
import numpy as np
import math
import os

# Distance to primary and secondary schools


In [ ]:
# read school and rental data
rental_data = pd.read_csv("../data/curated/rental_data_sf.csv")
school_data_s2 = pd.read_csv("../data/curated/school_data/school_data_s2.csv")
rental_data["coordinates"] = rental_data["coordinates"].apply(eval)
school_data_s2["coordinate"] = school_data_s2["coordinate"].apply(eval)
rental_data["SA2_CODE21"].value_counts()

In [ ]:
import requests
def calculate_distance_openroute(coordinate1, coordinate2):
    """ Function calculates distance between two coordinates using Openroutesrevice API"""
    lat1 = np.radians(coordinate1[0])
    lon1 = np.radians(coordinate1[1])
    lat2 = np.radians(coordinate2[0])
    lon2 = np.radians(coordinate2[1])
    import requests
    ORS_BASE_URL = "http://localhost:8080/ors/v2/directions/driving-car"
    url = f'{ORS_BASE_URL}?start={lon1},{lat1}&end={lon2},{lat2}'
    response = requests.get(url)
    print(response.reason)
    data = response.json()
    distance = data['features'][0]['properties']['segments'][0]['distance']
    return distance
    

In [ ]:
EARTH_RADIUS = 6371
def calculate_distance(coordinate1, coordinate2):
    """ Function calculates the haversine distance between two coordinates """
    # Convert latitude and longitude from degrees to radians
    lat1 = np.radians(coordinate1[0])
    lon1 = np.radians(coordinate1[1])
    lat2 = np.radians(coordinate2[0])
    lon2 = np.radians(coordinate2[1])

    # Haversine formula
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    distance = EARTH_RADIUS * c
    
    return distance
    
    
    

In [ ]:
def distance_closest_school(rental_data_coordinate, rental_data_sa2, school_data, new_col_names, school_types):
    """ Function finds the distance to the closest school from the rental property. If new_col_names list is of length 
    two it indicates that we are searching for closest secondary school and the function returns both distance to closest school 
    and boolean indicating if school is good. If new_col_names list is of length one, it indicates we are searching for primary schools and
    we only return distance to school """
    
    distance_dict = {}
    # Find closest school in SA2 area
    for index, school in school_data[(school_data["sa2_code_2021"] == rental_data_sa2) 
                                     & (school_data["school_type"].isin(school_types))].iterrows():
        distance_dict[school["coordinate"]] = calculate_distance(rental_data_coordinate, school["coordinate"])

    if distance_dict:
        # return school with shortest distance
        school_distance = min(distance_dict.items(), key=lambda x: x[1])[1]
        school_coordinate = min(distance_dict.items(), key=lambda x: x[1])[0]
        
        # for primary school just return distance to closest school
        if len(new_col_names) == 1:
            return pd.Series({new_col_names[0]: school_distance})
        # for secondary school return distance to closest school and if the school is top vce school
        elif len(new_col_names) == 2:
            # is school a top school
            top_school = ~(school_data[(school_data["coordinate"] == school_coordinate) & 
                                       (school_data["sa2_code_2021"] == rental_data_sa2)]["2022 ranking"].isna()).iloc[0]
            return pd.Series({new_col_names[0]: school_distance, 
                              new_col_names[1]: top_school})
    
    # school not found in SA2 area
    else:
        # Find closest school by comparing with all schools in victoria
        for index, school in school_data[school_data["school_type"].isin(school_types)].iterrows():
            distance_dict[school["coordinate"]] = [
            calculate_distance(rental_data_coordinate, school["coordinate"]), school["sa2_code_2021"]]
       
        if distance_dict:
            # return school with shortest distance 
            school_distance = min(distance_dict.items(), key=lambda x: x[1][0])[1][0]
            school_sa2_code_2021 = min(distance_dict.items(), key=lambda x: x[1][0])[1][1]
            school_coordinate = min(distance_dict.items(), key=lambda x: x[1][0])[0]
            # for primary school just return distance to closest school
            if len(new_col_names) == 1:
                return pd.Series({new_col_names[0]: school_distance})
            # for secondary school return distance to closest school and if the school is top vce school
            elif len(new_col_names) == 2:
                # is school a top school
                top_school = ~(school_data[(school_data["coordinate"] == school_coordinate) & 
                                       (school_data["sa2_code_2021"] == school_sa2_code_2021)]["2022 ranking"].isna()).iloc[0]
                return pd.Series({new_col_names[0]: school_distance, 
                                new_col_names[1]: top_school})
        
# find closest primary school    
primary_school = ["primary", "pri/sec"]
new_col_names_primary = ["distance primary school"]
new_columns_primary = rental_data.apply(
    lambda x: distance_closest_school(x["coordinates"], x["SA2_CODE21"], school_data_s2, new_col_names_primary, primary_school), axis=1)
rental_data = pd.concat([rental_data, new_columns_primary], axis=1)

# find closest secondary school
secondary_school = ["secondary"]
new_col_names_secondary = ["distance secondary school", "is closest secondary school good"]
new_columns_secondary = rental_data.apply(
    lambda x: distance_closest_school(x["coordinates"], x["SA2_CODE21"], school_data_s2, new_col_names_secondary, secondary_school), axis=1)
rental_data = pd.concat([rental_data, new_columns_secondary], axis=1)

# Find closest University Campus

In [ ]:
university_sa2 = pd.read_csv("../data/curated/school_data/university_location_curated_s2.csv")
university_sa2["coordinate"] = university_sa2["coordinate"].apply(eval)

In [ ]:
def distance_closest_university(rental_data_coordinate, rental_data_sa2, new_col_names, university_data):
    """ Function finds the distance to the closest university from the rental property """

    distance_dict = {}
    # Find closest university in SA2 area
    for index, university in university_data[university_data["sa2_code_2021"] == rental_data_sa2].iterrows():
        university_campus_name = university["institution"] + "-" + university["campus_name"]
        distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])
   
    # check with universities without SA2 code
    for index, university in university_data[university_data["sa2_code_2021"].isna()].iterrows():
        university_campus_name = university["institution"] + "-" + university["campus_name"]
        distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])


    if distance_dict:
        # return closest campus
        return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                          new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})
    
    else:
        # Find closest university by comparing witha all universities in victoria
        for index, university in university_data.iterrows():
            university_campus_name = university["institution"] + "-" + university["campus_name"]
            distance_dict[university_campus_name] = calculate_distance(rental_data_coordinate, university["coordinate"])
        if distance_dict:
            # return closest campus
            return pd.Series({new_col_names[0]: min(distance_dict.items(), key=lambda x: x[1])[0],
                            new_col_names[1]: min(distance_dict.items(), key=lambda x: x[1])[1]})

In [ ]:
new_col_names = ["closest university campus", "distance university campus"]
new_col_university = rental_data.apply(
    lambda x: distance_closest_university(x["coordinates"], x["SA2_CODE21"], new_col_names, university_sa2), axis=1)
rental_data = pd.concat([rental_data, new_col_university], axis=1)

In [ ]:
rental_data

In [ ]:
rental_data.to_csv("../data/curated/rental_data_distance_uni_school_sf.csv", index=False)